# Notebook 3 — Search Engines

## What this notebook builds
Two completely different search systems on the same 1,000 articles:

| System | Method | Understands meaning? |
|---|---|---|
| BM25 | Keyword matching | No — exact words only |
| FAISS | Vector similarity | Yes — meaning and context |

Building both lets us compare them side by side in Notebook 4.

---

## How each system works

**BM25 (keyword search)**
Ranks articles by how often your search words appear in them,
adjusted for how rare those words are and how long the article is.
It has no concept of meaning — "car" and "automobile" are completely
unrelated to it.

**FAISS (semantic search)**
Converts your query into a 384-number vector using the same model
that embedded the articles. Then finds the articles whose vectors
are closest in meaning — regardless of exact word choice.

---

## Key concept: why we need FAISS at all
With 1,000 articles, comparing a query against every article one by
one is fast enough. But at 1 million articles it would take seconds.
FAISS organises vectors into a structure that skips large portions
of the search space — making it millisecond-fast at any scale.

---

## Inputs and outputs

| File | Direction | Contents |
|---|---|---|
| `ag_news_1000.csv` | Input | 1,000 articles |
| `embeddings.npy` | Input | 1,000 × 384 mean-centered vectors |
| `mean_vector.npy` | Input | Shape (384,) — applied to every query |
| `faiss_index.bin` | Output | Saved FAISS index |
| `bm25_corpus.pkl` | Output | Saved BM25 model |

# Import Libraries

In [1]:
# Import and setup
# Crash prevention — must be first 
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # prevent OpenMP conflict
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"       # silence TF warnings

import warnings
warnings.filterwarnings("ignore")

# Working directory 
BASE_PATH = r"C:\Users\ishaa\OneDrive\Documents\Projects\Semantic Search Engine"
os.chdir(BASE_PATH)

# Libraries 
import numpy as np
import pandas as pd
import faiss                               # vector similarity search
import pickle                              # save/load Python objects to disk
import time                                # measure search speed

from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi            # keyword search algorithm
from sklearn.metrics.pairwise import cosine_similarity

print(f"Working directory : {os.getcwd()}")
print(f"numpy   : {np.__version__}")
print(f"pandas  : {pd.__version__}")
print("✅ Environment ready.")


Working directory : C:\Users\ishaa\OneDrive\Documents\Projects\Semantic Search Engine
numpy   : 1.26.4
pandas  : 2.3.3
✅ Environment ready.


# Load all inputs

In [2]:
# Load dataset 
df    = pd.read_csv("ag_news_1000.csv")
texts = df["content"].tolist()    # plain list of 1000 article strings

# Load embeddings and mean vector 
# These were saved at the end of Notebook 2.
# embeddings   : (1000, 384) mean-centered matrix
# mean_vector  : (384,) — must subtract from every query vector
embeddings  = np.load("embeddings.npy")
mean_vector = np.load("mean_vector.npy")

# Load the same model used in Notebook 2 
# We need it to embed search queries (not articles — those are done).
# Loads from local cache — should take only 2–3 seconds.
print("Loading model from cache...")
model = SentenceTransformer("all-MiniLM-L12-v2")

print(f"\nDataset    : {len(df)} articles, {list(df.columns)}")
print(f"Embeddings : {embeddings.shape}")
print(f"Mean vector: {mean_vector.shape}")
print(f"Model      : all-MiniLM-L12-v2 ✅")

Loading model from cache...

Dataset    : 1000 articles, ['content', 'category']
Embeddings : (1000, 384)
Mean vector: (384,)
Model      : all-MiniLM-L12-v2 ✅


# Build the FAISS Index
## What is a FAISS index?

A `FAISS index` is an organised data structure that stores all your
embedding vectors and allows extremely fast similarity search.

Think of it like a library catalogue vs searching every book shelf:

**Without FAISS:** compare query against all 1000 articles one by one  
**With FAISS:** jump directly to the most relevant section

### Index type: `IndexFlatIP`

- **Flat** = store all vectors exactly (no compression or approximation)
- **IP** = Inner Product — for normalised vectors this equals
  cosine similarity, which is what we want

### Why normalise?

Raw vectors have different lengths (magnitudes).

Normalising scales every vector to length **1.0** so that
the inner product equals cosine similarity exactly.

Without normalisation, longer vectors would dominate results
regardless of actual meaning similarity.

In [3]:
print("Building FAISS index...")
print()

# Step 1: get embedding dimension (384)
dim = embeddings.shape[1]
print(f"  Embedding dimension : {dim}")

# Step 2: create the index
# IndexFlatIP = exact search using inner product (cosine similarity)
index = faiss.IndexFlatIP(dim)
print(f"  Index type          : IndexFlatIP (exact cosine search)")

# Step 3: normalise embeddings
# faiss.normalize_L2() scales every vector so its length = 1.0
# It modifies the array IN PLACE so we work on a copy
embeddings_norm = embeddings.copy().astype(np.float32)
faiss.normalize_L2(embeddings_norm)
print(f"  Normalisation       : applied (L2)")

# Step 4: add normalised vectors to the index
# index.add() loads all 1000 vectors into the FAISS data structure
index.add(embeddings_norm)
print(f"  Vectors added       : {index.ntotal}")

print()
print("✅ FAISS index built successfully.")
print(f"   Total vectors in index : {index.ntotal}")
print(f"   Search dimension       : {dim}")

Building FAISS index...

  Embedding dimension : 384
  Index type          : IndexFlatIP (exact cosine search)
  Normalisation       : applied (L2)
  Vectors added       : 1000

✅ FAISS index built successfully.
   Total vectors in index : 1000
   Search dimension       : 384


# Build the BM25 index
## What is BM25?

`BM25` stands for **"Best Match 25"** — a keyword ranking algorithm
that has been the standard in search engines for 30+ years.

Google used it before neural search. Elasticsearch still uses it.

### How it scores an article for a query

1. **Term frequency**: how often does the query word appear in this article?
2. **Inverse document frequency**: how rare is this word across all 1000 articles?
   - Rare words like `"nasdaq"` are more informative than common words like `"the"`
3. **Length penalty**: long articles get penalised slightly
   - Appearing 3 times in a short article beats 3 times in a long one

### What BM25 cannot do

It has **NO concept of meaning**.

- `"car"` and `"automobile"` are completely unrelated to BM25.
- `"heart attack"` and `"cardiac arrest"` are unrelated.
- It only matches the exact characters you type.

### Tokenisation

Tokenisation means splitting text into individual words.

```text
"The stock market crashed"
→ ["the", "stock", "market", "crashed"]
```

BM25 works on these word lists, not on raw strings.

In [4]:
print("Building BM25 index...")
print()

# Step 1: tokenise every article into a list of lowercase words
# .lower()  converts to lowercase so "Stock" and "stock" match
# .split()  splits on whitespace — simple but effective tokenisation
tokenised_corpus = [text.lower().split() for text in texts]

print(f"  Articles tokenised  : {len(tokenised_corpus)}")
print(f"  Example tokenisation:")
print(f"  Original : {texts[0][:80]}...")
print(f"  Tokenised: {tokenised_corpus[0][:12]}...")
print()

# Step 2: build the BM25 model
# BM25Okapi is the most widely used BM25 variant
# It takes the tokenised corpus and computes term frequencies
# and inverse document frequencies for all words
bm25 = BM25Okapi(tokenised_corpus)

print(f"  Vocabulary size     : {len(bm25.idf)} unique words")
print()
print("✅ BM25 index built successfully.")

Building BM25 index...

  Articles tokenised  : 1000
  Example tokenisation:
  Original : Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall ...
  Tokenised: ['wall', 'st.', 'bears', 'claw', 'back', 'into', 'the', 'black', '(reuters)', 'reuters', '-', 'short-sellers,']...

  Vocabulary size     : 10037 unique words

✅ BM25 index built successfully.


# Define search functions
Defines the two core search functions. Both accept the same inputs and return results in the same format — making direct comparison straightforward. The semantic search function applies mean centering and normalisation to every query before searching, matching exactly what was done to the article embeddings.

In [5]:
# Two clean search functions — one per system 
# These functions are the core of the project.
# Both take the same inputs and return the same output format
# so they can be compared directly in Notebook 4 and the app.

def semantic_search(query, top_k=5):
    """
    Search using meaning (FAISS + sentence transformer).

    Steps:
    1. Embed the query using the same model used for articles
    2. Apply mean centering (subtract mean_vector)
    3. Normalise to unit length (required for IndexFlatIP)
    4. Ask FAISS for the top_k most similar vectors
    5. Return matching articles with their similarity scores

    Args:
        query  : search string typed by the user
        top_k  : number of results to return (default 5)

    Returns:
        list of dicts with keys: rank, score, category, text
    """
    # Step 1: embed the query → shape (384,)
    q_vec = model.encode(query, convert_to_numpy=True)

    # Step 2: apply mean centering — same operation done to articles
    # Without this, query lives in a different region of vector space
    # than the articles → poor results
    q_vec = q_vec - mean_vector

    # Step 3: normalise to unit length, reshape to (1, 384) for FAISS
    q_vec = q_vec.astype(np.float32).reshape(1, -1)
    faiss.normalize_L2(q_vec)

    # Step 4: FAISS search
    # index.search() returns two arrays:
    #   scores  : similarity scores, shape (1, top_k)
    #   indices : article indices,   shape (1, top_k)
    # [0] takes the first (and only) query row from each
    scores, indices = index.search(q_vec, top_k)
    scores  = scores[0]     # shape (top_k,)
    indices = indices[0]    # shape (top_k,)

    # Step 5: build results list
    results = []
    for rank, (idx, score) in enumerate(zip(indices, scores), 1):
        results.append({
            "rank"    : rank,
            "score"   : round(float(score), 4),
            "category": df.loc[idx, "category"],
            "text"    : df.loc[idx, "content"],
            "index"   : int(idx),
        })
    return results


def keyword_search(query, top_k=5):
    """
    Search using exact keywords (BM25).

    Steps:
    1. Tokenise the query into lowercase words
    2. Ask BM25 to score all 1000 articles against those words
    3. Sort by score descending
    4. Return top_k results

    Args:
        query  : search string typed by the user
        top_k  : number of results to return (default 5)

    Returns:
        list of dicts with keys: rank, score, category, text
    """
    # Step 1: tokenise query the same way as the corpus
    query_tokens = query.lower().split()

    # Step 2: BM25 scores all 1000 articles simultaneously
    # Returns a numpy array of 1000 scores, one per article
    scores = bm25.get_scores(query_tokens)

    # Step 3: argsort() returns indices sorted lowest→highest
    # [-top_k:]  takes the last top_k (highest scores)
    # [::-1]     reverses to highest→lowest
    top_indices = np.argsort(scores)[-top_k:][::-1]

    # Step 4: build results list
    results = []
    for rank, idx in enumerate(top_indices, 1):
        results.append({
            "rank"    : rank,
            "score"   : round(float(scores[idx]), 4),
            "category": df.loc[idx, "category"],
            "text"    : df.loc[idx, "content"],
            "index"   : int(idx),
        })
    return results


print("✅ Both search functions defined.")
print()
print("semantic_search(query, top_k=5) — FAISS + sentence transformer")
print("keyword_search(query,  top_k=5) — BM25 keyword matching")

✅ Both search functions defined.

semantic_search(query, top_k=5) — FAISS + sentence transformer
keyword_search(query,  top_k=5) — BM25 keyword matching


# Test both search functions
Runs both search functions against 5 different queries and prints top 3 results from each side by side. Also measures speed in milliseconds.

In [6]:
# Run both systems on the same 5 queries 
# Purpose: visually confirm both systems work correctly
# and get a first look at where they agree and differ.

def print_results(results, system_name):
    """Print search results in a clean, readable format."""
    print(f"\n  {system_name}")
    print(f"  {'─' * 52}")
    for r in results:
        # Truncate article text for display
        preview = r["text"][:100].replace("\n", " ")
        print(f"  [{r['rank']}] ({r['category']:<12}) "
              f"score={r['score']:>7.4f} | {preview}...")


test_queries = [
    "stock market and investments",
    "football world cup",
    "iPhone smartphone release",
    "presidential election results",
    "NASA space exploration",
]

print("=" * 60)
print("SEARCH SYSTEM TEST — 5 QUERIES")
print("=" * 60)

for query in test_queries:
    print(f"\nQuery: '{query}'")

    # Run semantic search and measure time
    t0       = time.time()
    sem_res  = semantic_search(query, top_k=3)
    sem_time = (time.time() - t0) * 1000   # convert to milliseconds

    # Run keyword search and measure time
    t0       = time.time()
    kw_res   = keyword_search(query, top_k=3)
    kw_time  = (time.time() - t0) * 1000

    print_results(sem_res, f"Semantic  ({sem_time:.1f}ms)")
    print_results(kw_res,  f"Keyword   ({kw_time:.1f}ms)")
    print()

SEARCH SYSTEM TEST — 5 QUERIES

Query: 'stock market and investments'

  Semantic  (336.5ms)
  ────────────────────────────────────────────────────
  [1] (Business    ) score= 0.4294 | Art Looks Like Fine Investment for Funds  NEW YORK (Reuters) - Some mutual funds invest in stocks;  ...
  [2] (Business    ) score= 0.4283 | Oil and Economy Cloud Stocks' Outlook  NEW YORK (Reuters) - Soaring crude prices plus worries  about...
  [3] (Business    ) score= 0.4220 | Oil and Economy Cloud Stocks' Outlook  NEW YORK (Reuters) - Soaring crude prices plus worries  about...

  Keyword   (0.0ms)
  ────────────────────────────────────────────────────
  [1] (Business    ) score= 8.7824 | Veteran inventor in market float Trevor Baylis, the veteran inventor famous for creating the Freepla...
  [2] (Business    ) score= 8.1576 | Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices plus worries\about th...
  [3] (Business    ) score= 8.1576 | Oil and Economy Cloud Stocks' Outl

# Speed Benchmark
Measures average, minimum, and maximum search time for both systems over 50 queries. Semantic search is slower than BM25 because it runs the neural model on each query — BM25 is just arithmetic on pre-computed word frequencies.

In [7]:
# Speed benchmark — 50 repeated searches 
# A single search is too fast to measure accurately (< 1ms for FAISS).
# Running 50 searches and averaging gives a reliable speed estimate.
# This matters for the app — users expect instant results.

benchmark_queries = [
    "technology innovation",
    "sports championship",
    "business economy",
    "world politics",
    "stock market crash",
] * 10   # repeat 5 queries × 10 = 50 total searches

sem_times = []
kw_times  = []

for q in benchmark_queries:
    t0 = time.time()
    semantic_search(q, top_k=5)
    sem_times.append((time.time() - t0) * 1000)

    t0 = time.time()
    keyword_search(q, top_k=5)
    kw_times.append((time.time() - t0) * 1000)

print("Speed benchmark (50 queries each):")
print("─" * 40)
print(f"  Semantic search (FAISS)")
print(f"    Avg : {np.mean(sem_times):.2f} ms")
print(f"    Min : {np.min(sem_times):.2f} ms")
print(f"    Max : {np.max(sem_times):.2f} ms")
print()
print(f"  Keyword search (BM25)")
print(f"    Avg : {np.mean(kw_times):.2f} ms")
print(f"    Min : {np.min(kw_times):.2f} ms")
print(f"    Max : {np.max(kw_times):.2f} ms")
print()
print("Note: semantic search is slower because it runs the")
print("neural model on the query. BM25 is pure arithmetic.")

Speed benchmark (50 queries each):
────────────────────────────────────────
  Semantic search (FAISS)
    Avg : 56.59 ms
    Min : 39.91 ms
    Max : 78.01 ms

  Keyword search (BM25)
    Avg : 1.68 ms
    Min : 0.00 ms
    Max : 15.88 ms

Note: semantic search is slower because it runs the
neural model on the query. BM25 is pure arithmetic.


# Save both Indexes

In [8]:
# Save FAISS index 
# faiss.write_index() serialises the index to a binary file.
# faiss.read_index()  loads it back — index is immediately ready to search.
# This means the Streamlit app and Notebook 4 don't need to rebuild it.

faiss_path = "faiss_index.bin"
faiss.write_index(index, faiss_path)
print(f"FAISS index saved  : {faiss_path}")
print(f"  Vectors stored   : {index.ntotal}")
print(f"  File size        : {os.path.getsize(faiss_path)/1e6:.3f} MB")

# Save BM25 model 
# BM25 is a Python object — we use pickle to save it.
# pickle serialises any Python object to bytes and saves to disk.
# pickle.dump()  → save
# pickle.load()  → load back

bm25_path = "bm25_model.pkl"
with open(bm25_path, "wb") as f:   # "wb" = write binary
    pickle.dump(bm25, f)

print(f"\nBM25 model saved   : {bm25_path}")
print(f"  File size        : {os.path.getsize(bm25_path)/1e6:.3f} MB")

# Verify both saves by reloading 
print("\nVerifying saves...")

index_check = faiss.read_index(faiss_path)
with open(bm25_path, "rb") as f:   # "rb" = read binary
    bm25_check = pickle.load(f)

# Quick functional test on the reloaded objects
test_sem = semantic_search("technology news", top_k=1)
test_kw  = keyword_search("technology news",  top_k=1)

print(f"  FAISS reload     : {index_check.ntotal} vectors ✅")
print(f"  BM25  reload     : {len(bm25_check.idf)} vocab terms ✅")
print(f"  Semantic test    : '{test_sem[0]['text'][:60]}...' ✅")
print(f"  Keyword test     : '{test_kw[0]['text'][:60]}...' ✅")

print()
print("=" * 50)
print("Files saved — ready for Notebook 4 and app.py")
print("=" * 50)
print(f"  ag_news_1000.csv   — dataset")
print(f"  embeddings.npy     — article vectors")
print(f"  mean_vector.npy    — query centering")
print(f"  faiss_index.bin    — semantic search index")
print(f"  bm25_model.pkl     — keyword search model")

FAISS index saved  : faiss_index.bin
  Vectors stored   : 1000
  File size        : 1.536 MB

BM25 model saved   : bm25_model.pkl
  File size        : 0.501 MB

Verifying saves...
  FAISS reload     : 1000 vectors ✅
  BM25  reload     : 10037 vocab terms ✅
  Semantic test    : 'Sprint Broadens Its Vision (NewsFactor) NewsFactor - Sprint ...' ✅
  Keyword test     : ''Invisible' technology for Olympics Getting the technology i...' ✅

Files saved — ready for Notebook 4 and app.py
  ag_news_1000.csv   — dataset
  embeddings.npy     — article vectors
  mean_vector.npy    — query centering
  faiss_index.bin    — semantic search index
  bm25_model.pkl     — keyword search model


---

## Notebook 3 — Summary & Results

### What we built
Two fully functional search engines on 1,000 AG News articles,
both saved to disk and ready for Notebook 4 and the Streamlit app.

---

### Files created

| File | Size | Contents |
|---|---|---|
| `faiss_index.bin` | ~1.5 MB | FAISS index with 1,000 normalised vectors |
| `bm25_model.pkl` | ~1.5 MB | BM25 model with 10,037 vocabulary terms |

---

### How each system works

| | Semantic (FAISS) | Keyword (BM25) |
|---|---|---|
| Method | Vector similarity | Word frequency matching |
| Understands synonyms | Yes | No |
| Understands meaning | Yes | No |
| Speed (avg) | 56.59 ms | 1.68 ms |
| Speed ratio | — | 34× faster |

---

### Search results summary — 5 test queries

| Query | Semantic top category | Keyword top category | Agreement |
|---|---|---|---|
| stock market and investments | Business | Business | ✅ Same |
| football world cup | Technology* | Sports | ❌ Differ |
| iPhone smartphone release | Technology | Technology | ✅ Same |
| presidential election results | World | World | ✅ Same |
| NASA space exploration | Technology | Technology | ✅ Same |

*Semantic returned Madden NFL video game articles (Technology) because
they contain heavy football vocabulary — correct semantic match,
wrong category label. Demonstrates cross-category semantic retrieval.

---

### Key observations

**Where semantic search wins**
- "iPhone smartphone release" → found Nokia and Sprint phone articles
  without the word "iPhone" appearing anywhere in them
- "NASA space exploration" → top score of 0.6369, very high confidence

**Where keyword search wins**
- "football world cup" → correctly found Sports articles
  because exact terms "football" and "World Cup" appeared in text
- "presidential election results" → both systems agreed, but BM25
  was 28× faster for identical results

**Most interesting result**
The football query shows the core trade-off of this project:
semantic search found video game articles labelled Technology
because the content discusses football — the model read the meaning,
not the label. BM25 found actual sports results by matching words.
Neither is wrong — they answer different questions.

---

### Known limitations

| Limitation | Cause | Impact |
|---|---|---|
| Duplicate articles in results | AG News dataset quality | Cosmetic — same article appears twice |
| First query slower (~330ms) | Neural model warm-up | Only affects first search per session |
| Football query cross-category | Dataset labelling inconsistency | Educational — shows semantic behaviour |
| BM25 "release" mismatch | Word polysemy (one word, many meanings) | Classic keyword search failure |

---

### Next step
Open **`04_comparison.ipynb`** to run a structured comparison of both
systems across multiple query types and analyse where each method
wins, loses, and why.